# Assignment 1 — training pipeline

This notebook prepares the Ames sales for a linear model. It does not train yet.

`uv run python main.py train` runs every cell below with no manual steps.
The decisions are written up in the README under "Preprocessing".


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("AmesHousing.csv")

# Impossible values, fixed with the same rule on every row.
df.loc[df["Garage Yr Blt"] == 2207, "Garage Yr Blt"] = 2007
remod_before_built = df["Year Remod/Add"] < df["Year Built"]
df.loc[remod_before_built, "Year Remod/Add"] = df.loc[remod_before_built, "Year Built"]
# A remodel year after the sale is not known yet.
future_remod = df["Year Remod/Add"] > df["Yr Sold"]
df.loc[future_remod, "Year Remod/Add"] = df.loc[future_remod, "Yr Sold"]

# Three partial sales of houses above 4,000 sq ft. De Cock: these prices are not market prices.
# The two other houses above 4,000 sq ft are kept: their prices match each other.
partial_outlier = (df["Gr Liv Area"] > 4000) & (df["Sale Condition"] == "Partial")
df = df.loc[~partial_outlier].copy()

# A blank means the house does not have that part.
no_basement = df["Bsmt Qual"].isna()
for col in ["Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]:
    df.loc[no_basement, col] = "None"
for col in ["BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF", "Total Bsmt SF", "Bsmt Full Bath", "Bsmt Half Bath"]:
    df.loc[no_basement, col] = 0

no_garage = df["Garage Type"].isna()
for col in ["Garage Type", "Garage Finish", "Garage Qual", "Garage Cond"]:
    df.loc[no_garage, col] = "None"
df.loc[no_garage, "Garage Area"] = 0

df["Mas Vnr Type"] = df["Mas Vnr Type"].fillna("None")
no_veneer = df["Mas Vnr Type"].eq("None") & df["Mas Vnr Area"].isna()
df.loc[no_veneer, "Mas Vnr Area"] = 0

for col in ["Alley", "Fireplace Qu", "Fence"]:
    df[col] = df[col].fillna("None")

df["MS SubClass"] = df["MS SubClass"].astype(str)
df["Mo Sold"] = df["Mo Sold"].astype(str).str.zfill(2)

# Price a sale using only earlier sales. 2010 stops in July.
train = df[df["Yr Sold"] <= 2008].copy()
val = df[df["Yr Sold"] == 2009].copy()
test = df[df["Yr Sold"] == 2010].copy()

# Fill true gaps from the training years only.
frontage_by_neighborhood = train.groupby("Neighborhood")["Lot Frontage"].median()
frontage_median = train["Lot Frontage"].median()
veneer_area_by_type = train.groupby("Mas Vnr Type")["Mas Vnr Area"].median()
veneer_area_median = train["Mas Vnr Area"].median()

detached = train["Garage Type"].eq("Detchd")
detached_fill = {
    "Garage Finish": train.loc[detached, "Garage Finish"].mode().iloc[0],
    "Garage Qual": train.loc[detached, "Garage Qual"].mode().iloc[0],
    "Garage Cond": train.loc[detached, "Garage Cond"].mode().iloc[0],
    "Garage Yr Blt": train.loc[detached, "Garage Yr Blt"].median(),
    "Garage Area": train.loc[detached, "Garage Area"].median(),
}

with_basement = train["Bsmt Qual"].ne("None")
basement_fill = {
    "Bsmt Exposure": train.loc[with_basement, "Bsmt Exposure"].mode().iloc[0],
    "BsmtFin Type 2": train.loc[with_basement, "BsmtFin Type 2"].mode().iloc[0],
}
electrical_mode = train["Electrical"].mode().iloc[0]


def fill_gaps(frame):
    frame = frame.copy()
    frame["Lot Frontage"] = frame["Lot Frontage"].fillna(
        frame["Neighborhood"].map(frontage_by_neighborhood)
    )
    frame["Lot Frontage"] = frame["Lot Frontage"].fillna(frontage_median)
    frame["Mas Vnr Area"] = frame["Mas Vnr Area"].fillna(
        frame["Mas Vnr Type"].map(veneer_area_by_type)
    )
    frame["Mas Vnr Area"] = frame["Mas Vnr Area"].fillna(veneer_area_median)
    for col, value in detached_fill.items():
        frame[col] = frame[col].fillna(value)
    for col, value in basement_fill.items():
        frame[col] = frame[col].fillna(value)
    frame["Electrical"] = frame["Electrical"].fillna(electrical_mode)
    return frame


train = fill_gaps(train)
val = fill_gaps(val)
test = fill_gaps(test)


def add_ages(frame):
    frame = frame.copy()
    frame["Age"] = frame["Yr Sold"] - frame["Year Built"]
    frame["RemodAge"] = frame["Yr Sold"] - frame["Year Remod/Add"]
    # 0 when there is no garage. A raw year would invent a construction date.
    frame["GarageAge"] = 0.0
    has_garage = frame["Garage Type"].ne("None")
    frame.loc[has_garage, "GarageAge"] = (
        frame.loc[has_garage, "Yr Sold"] - frame.loc[has_garage, "Garage Yr Blt"]
    )
    return frame


train = add_ages(train)
val = add_ages(val)
test = add_ages(test)

QUALITY = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
FINISH = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
ORDINAL = {
    "Exter Qual": QUALITY,
    "Exter Cond": QUALITY,
    "Bsmt Qual": QUALITY,
    "Bsmt Cond": QUALITY,
    "Heating QC": QUALITY,
    "Kitchen Qual": QUALITY,
    "Fireplace Qu": QUALITY,
    "Garage Qual": QUALITY,
    "Garage Cond": QUALITY,
    "Bsmt Exposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFin Type 1": FINISH,
    "BsmtFin Type 2": FINISH,
    "Functional": {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7},
    "Garage Finish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Lot Shape": {"IR3": 0, "IR2": 1, "IR1": 2, "Reg": 3},
    "Land Slope": {"Gtl": 0, "Mod": 1, "Sev": 2},
    "Paved Drive": {"N": 0, "P": 1, "Y": 2},
    "Fence": {"None": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
    "Central Air": {"N": 0, "Y": 1},
}

NOMINAL = [
    "MS SubClass", "MS Zoning", "Alley", "Lot Config", "Land Contour",
    "Neighborhood", "Condition 1", "Bldg Type", "House Style", "Roof Style",
    "Exterior 1st", "Mas Vnr Type", "Foundation", "Garage Type", "Electrical",
    "Mo Sold",
]

DROP = [
    "Order", "PID", "Street", "Utilities", "Condition 2", "Roof Matl", "Heating",
    "Exterior 2nd", "Pool QC", "Misc Feature", "Sale Type", "Sale Condition",
    "1st Flr SF", "2nd Flr SF", "Low Qual Fin SF", "Bsmt Unf SF", "Garage Cars",
    "Year Built", "Year Remod/Add", "Garage Yr Blt", "SalePrice",
]


def encode_ordinal(frame):
    frame = frame.copy()
    for col, mapping in ORDINAL.items():
        frame[col] = frame[col].map(mapping)
    return frame


train = encode_ordinal(train)
val = encode_ordinal(val)
test = encode_ordinal(test)

# Fewer than 10 training rows, and the level is renamed Other in later years too.
for col in NOMINAL:
    common = set(train[col].value_counts().loc[lambda s: s >= 10].index)
    for frame in (train, val, test):
        frame[col] = frame[col].where(frame[col].isin(common), "Other")


def matrix(frame):
    features = frame.drop(columns=DROP)
    still_missing = features.columns[features.isna().any()].tolist()
    if still_missing:
        raise ValueError(f"Missing values left in {still_missing}")
    y = np.log(frame["SalePrice"].to_numpy(dtype=float))
    return features, y


X_train, y_train = matrix(train)
X_val, y_val = matrix(val)
X_test, y_test = matrix(test)

text_left = X_train.select_dtypes(exclude="number").columns.difference(NOMINAL)
if len(text_left) > 0:
    raise ValueError(f"Columns not encoded: {text_left.tolist()}")

# Alphabetical order, then leave out the first level. The intercept stands in for it.
for col in NOMINAL:
    categories = sorted(X_train[col].astype(str).unique())
    for frame in (X_train, X_val, X_test):
        frame[col] = pd.Categorical(frame[col], categories=categories)


def align(features):
    numeric = features.drop(columns=NOMINAL).reset_index(drop=True)
    dummies = pd.get_dummies(features[NOMINAL], drop_first=True, dtype=float)
    return pd.concat([numeric, dummies.reset_index(drop=True)], axis=1)


X_train = align(X_train)
X_val = align(X_val)
X_test = align(X_test)
if not (list(X_train.columns) == list(X_val.columns) == list(X_test.columns)):
    raise ValueError("Train, validation, and test columns differ")
feature_names = list(X_train.columns)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
X_val = pd.DataFrame(scaler.transform(X_val), columns=feature_names)
X_test = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

if X_train.isna().any().any() or X_val.isna().any().any() or X_test.isna().any().any():
    raise ValueError("Missing values left after scaling")

print(f"Rows: train {len(y_train)}, validation {len(y_val)}, test {len(y_test)}")
print(f"Features: {X_train.shape[1]}")
print(f"Training geometric mean price: ${np.exp(y_train.mean()):,.0f}")
